# Step 07 / Phase A-D: Multi-Label Fault Classification — Canonical, V6

**Dataset**: V6 (`cooked_data_v6/features_engineered_v6.pkl`) — 2,109 events, 766 features,
leakage-safe (scaler/PCA fit on each approach's own train fold).

Per `spiral2-pipeline-context`: Phases A/B/C/D are **four competing modeling approaches** to the
same multi-label fault-classification problem (not sequential pipeline steps) — Binary Relevance,
Classifier Chains, Label Powerset, and deep multi-head models (CNN/Transformer/CNN-LSTM/MLP).
Step 07's own script (`prepare_07_phase2_multilabel.py`) is a simpler standalone
`MultiOutputClassifier(RandomForest)` baseline, included here for completeness.

**Why this notebook was rebuilt (2026-09-05)**: the report currently claims Multi-Head CNN is the
best Phase 07 model (93.5% Micro-F1 / 0.892 Macro-F1), contradicting every current notebook's own
prior analysis (which already found Label Powerset best) — see `NOTEBOOK_AUDIT_2026-09-03.md`.
That prior analysis was itself run on leakage-affected numbers (scaler/PCA fit on the full dataset
before any split — see `spiral2_open_methodology_questions` memory item 5). **This notebook
re-verifies the same conclusion on leakage-safe V6 numbers, now testing all four Phase-D
architectures (not just CNN)**, so the claim can be corrected with full confidence.

Pipeline runs (SLURM jobs, all leakage-safe, corrected V6 dataset, 2026-09-05):
- Step 07 (RF baseline): job 57924850
- Phase A (BR-RF / CC-XGBoost): job 57924852
- Phase B (CC-XGBoost): job 57924853
- Phase C (Label Powerset): job 57924854
- Phase D (CNN/MLP/Transformer/CNN-LSTM): jobs 57924855, 57931121, 57931122, 57931123

This notebook loads each job's saved results; it does not retrain models.

In [1]:
import pickle
import pandas as pd
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))
from utilities.reporting.manifest import save_manifest

BASE = Path('/sps/m4cast/_spiral2_data/_llrf_data/cooked_data_v6')

def load(rel):
    with open(BASE / rel, 'rb') as f:
        return pickle.load(f)

step07 = load('step_07_phase2_v6/multilabel_triggers.pkl')
phaseA = load('step_07_phaseA_v6_br/phaseA_summary.pkl')
phaseB = load('step_08_phaseB_cc/phaseB_summary.pkl')
phaseC = load('step_09_phaseC_lp/phaseC_summary.pkl')
phaseD = {}
for arch in ['cnn', 'mlp', 'transformer', 'cnn_lstm']:
    phaseD[arch] = load(f'step_10_phaseD/{arch}/phaseD_summary_{arch}.pkl')

print('Loaded step 07 + all Phase A/B/C/D results.')

Loaded step 07 + all Phase A/B/C/D results.


In [2]:
rows = [
    {'Approach': 'Step 07 baseline (MultiOutput RF)',    'Micro-F1': None, 'Macro-F1': None, 'Hamming': step07['hamming_loss']},
    {'Approach': 'Phase A: Binary Relevance (RF)',        'Micro-F1': phaseA['br']['micro_f1'], 'Macro-F1': phaseA['br']['macro_f1'], 'Hamming': phaseA['br']['hamming_loss']},
    {'Approach': 'Phase A: Classifier Chain (XGBoost)',   'Micro-F1': phaseA['cc']['micro_f1'], 'Macro-F1': phaseA['cc']['macro_f1'], 'Hamming': phaseA['cc']['hamming_loss']},
    {'Approach': 'Phase B: Classifier Chain (XGBoost)',   'Micro-F1': phaseB['cc']['micro_f1'], 'Macro-F1': phaseB['cc']['macro_f1'], 'Hamming': phaseB['cc']['hamming_loss']},
    {'Approach': 'Phase C: Label Powerset (XGBoost)',     'Micro-F1': phaseC['lp']['micro_f1'], 'Macro-F1': phaseC['lp']['macro_f1'], 'Hamming': phaseC['lp']['hamming_loss']},
    {'Approach': 'Phase D: Multi-Head CNN',               'Micro-F1': phaseD['cnn']['best']['metrics']['micro_f1'], 'Macro-F1': phaseD['cnn']['best']['metrics']['macro_f1'], 'Hamming': phaseD['cnn']['best']['metrics']['hamming']},
    {'Approach': 'Phase D: Multi-Head MLP',               'Micro-F1': phaseD['mlp']['best']['metrics']['micro_f1'], 'Macro-F1': phaseD['mlp']['best']['metrics']['macro_f1'], 'Hamming': phaseD['mlp']['best']['metrics']['hamming']},
    {'Approach': 'Phase D: Multi-Head Transformer',       'Micro-F1': phaseD['transformer']['best']['metrics']['micro_f1'], 'Macro-F1': phaseD['transformer']['best']['metrics']['macro_f1'], 'Hamming': phaseD['transformer']['best']['metrics']['hamming']},
    {'Approach': 'Phase D: Multi-Head CNN-LSTM',          'Micro-F1': phaseD['cnn_lstm']['best']['metrics']['micro_f1'], 'Macro-F1': phaseD['cnn_lstm']['best']['metrics']['macro_f1'], 'Hamming': phaseD['cnn_lstm']['best']['metrics']['hamming']},
]
df = pd.DataFrame(rows).sort_values('Micro-F1', ascending=False, na_position='last').reset_index(drop=True)
df

,Approach,Micro-F1,Macro-F1,Hamming
0,Phase C: Label Powerset (XGBoost),0.929630,0.801315,0.010281
1,Phase D: Multi-Head CNN,0.905747,0.623598,0.013879
2,Phase A: Classifier Chain (XGBoost),0.898734,0.718568,0.014444
3,Phase D: Multi-Head MLP,0.898148,0.738059,0.014895
4,Phase A: Binary Relevance (RF),0.895765,0.665617,0.014444
5,Phase B: Classifier Chain (XGBoost),0.891791,0.659661,0.015693
6,Phase D: Multi-Head Transformer,0.873508,0.482717,0.017942
7,Phase D: Multi-Head CNN-LSTM,0.844548,0.572537,0.022681
8,Step 07 baseline (MultiOutput RF),NaN,NaN,0.014669


## Conclusion

**Label Powerset (Phase C) is the best-performing approach by both Micro-F1 and Macro-F1**,
across all four deep multi-head architectures tested (CNN, MLP, Transformer, CNN-LSTM) as well as
Binary Relevance and both Classifier Chain implementations. This directly contradicts the report's
current claim that Multi-Head CNN is best — that claim should be corrected. All four Phase D
architectures land within a similar, lower range (Micro-F1 0.82-0.91, Macro-F1 0.48-0.74),
consistent with deep multi-head models not having an advantage over classical multi-label
strategies at this dataset size (~2,100 events).

In [3]:
best_row = df.iloc[0]
metrics = {}
for _, row in df.iterrows():
    slug = row['Approach'].lower().replace(' ', '_').replace(':', '').replace('(', '').replace(')', '').replace('-', '_')
    if pd.notna(row['Micro-F1']):
        metrics[f'{slug}_micro_f1'] = {'value': float(row['Micro-F1']), 'fmt': '.1%', 'label': f"{row['Approach']} Micro-F1"}
        metrics[f'{slug}_macro_f1'] = {'value': float(row['Macro-F1']), 'fmt': '.1%', 'label': f"{row['Approach']} Macro-F1"}
    metrics[f'{slug}_hamming'] = {'value': float(row['Hamming']), 'fmt': '.4f', 'label': f"{row['Approach']} Hamming loss"}
metrics['best_approach'] = {'value': best_row['Approach'], 'fmt': None, 'label': 'Best-performing multi-label approach'}

manifest = save_manifest(
    phase='07_multilabel_phaseABCD',
    metrics=metrics,
    pipeline_run={
        'dataset_version': 'V6',
        'dataset_path': '/sps/m4cast/_spiral2_data/_llrf_data/cooked_data_v6/features_engineered_v6.pkl',
        'slurm_job': '57924850,57924852,57924853,57924854,57924855,57931121,57931122,57931123',
        'script': 'pipeline/00_scripts/prepare_07_phase2_multilabel.py + phaseA_br_test.py + phaseB_cc_test.py + phaseC_lp_test.py + phaseD_train.py',
    },
    meta={
        'correction': 'Report claims Multi-Head CNN is best Phase 07 model (93.5% Micro-F1 / '
                      '0.892 Macro-F1) -- FALSIFIED by these leakage-safe V6 numbers. Label '
                      'Powerset is best; all 4 Phase D architectures underperform it.',
    },
)
manifest

{'phase': '07_multilabel_phaseABCD',
 'generated_at': '2026-09-05T09:35:36.471225+00:00',
 'pipeline_run': {'dataset_version': 'V6',
  'dataset_path': '/sps/m4cast/_spiral2_data/_llrf_data/cooked_data_v6/features_engineered_v6.pkl',
  'slurm_job': '57924850,57924852,57924853,57924854,57924855,57931121,57931122,57931123',
  'script': 'pipeline/00_scripts/prepare_07_phase2_multilabel.py + phaseA_br_test.py + phaseB_cc_test.py + phaseC_lp_test.py + phaseD_train.py'},
 'metrics': {'phase_c_label_powerset_xgboost_micro_f1': {'value': 0.9296296296296296,
   'fmt': '.1%',
   'label': 'Phase C: Label Powerset (XGBoost) Micro-F1',
   'display': '93.0%'},
  'phase_c_label_powerset_xgboost_macro_f1': {'value': 0.8013150061417191,
   'fmt': '.1%',
   'label': 'Phase C: Label Powerset (XGBoost) Macro-F1',
   'display': '80.1%'},
  'phase_c_label_powerset_xgboost_hamming': {'value': 0.010281385281385282,
   'fmt': '.4f',
   'label': 'Phase C: Label Powerset (XGBoost) Hamming loss',
   'display': '0.